# Semana 3: Tarea

**Valoración DCF de tu empresa** (la misma que elegiste en la tarea de la semana 2)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/03_dcf_fcff_fcfe/clase03_tarea.ipynb)

**Entrega: hasta el lunes de la semana siguiente, vía `git commit` + `push` en tu fork** (`03_dcf_fcff_fcfe/clase03_tarea.ipynb`)

Total: 20 puntos (18 de contenido y 2 de forma: el notebook corre de inicio a fin y la entrega llega a tiempo).

## Parte 1: FCFF histórico por dos rutas (4 pts)

Con los estados financieros de tu empresa (`yfinance`), calcula el FCFF de los últimos 3 a 4 años por la ruta del EBIT y por la ruta del CFO, en una tabla comparativa. Comenta en una o dos oraciones por qué las rutas no coinciden exactamente.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from utils.finanzas import (capm, wacc, fcff_desde_ebit, fcff_desde_cfo,
                            fcfe_desde_fcff, valor_crecimiento_constante)

TICKER = "PG"
tk = yf.Ticker(TICKER)
print("Empresa:", tk.info.get("longName", TICKER))

def fila(df, *nombres):
    """Devuelve la primera fila disponible entre los nombres candidatos."""
    for n in nombres:
        if n in df.index:
            return df.loc[n]
    raise KeyError(f"Ninguna de estas filas existe: {nombres}")

est = tk.income_stmt
bal = tk.balance_sheet
cf  = tk.cashflow

ebit    = fila(est, "EBIT", "Operating Income")
interes = fila(est, "Interest Expense").abs()
impuesto = fila(est, "Tax Provision")
util_at  = fila(est, "Pretax Income")
dep     = fila(cf, "Depreciation And Amortization", "Depreciation Amortization Depletion", "Depreciation")
capex   = fila(cf, "Capital Expenditure").abs()
d_wc    = -fila(cf, "Change In Working Capital")   # signo: incremento de WC consume caja
cfo     = fila(cf, "Operating Cash Flow")

t_efectiva = float((impuesto / util_at).iloc[0])
print(f"Tasa efectiva de impuestos (ultimo anio): {t_efectiva:.1%}")

resumen = pd.DataFrame({"EBIT": ebit, "Dep": dep, "Capex": capex,
                        "WCInv": d_wc, "Interes": interes, "CFO": cfo}).dropna() / 1e6
resumen = resumen.sort_index(ascending=False)   # el anio mas reciente primero: .iloc[0] = ultimo anio
resumen.round(0)

anios = resumen.index
fcff_ebit = pd.Series(
    [fcff_desde_ebit(resumen.loc[a, "EBIT"], t_efectiva, resumen.loc[a, "Dep"],
                     resumen.loc[a, "Capex"], resumen.loc[a, "WCInv"]) for a in anios],
    index=anios, name="FCFF (ruta EBIT)")

fcff_cfo = pd.Series(
    [fcff_desde_cfo(resumen.loc[a, "CFO"], resumen.loc[a, "Interes"],
                    t_efectiva, resumen.loc[a, "Capex"]) for a in anios],
    index=anios, name="FCFF (ruta CFO)")

comparacion = pd.concat([fcff_ebit, fcff_cfo], axis=1)
comparacion["diferencia %"] = (fcff_ebit / fcff_cfo - 1) * 100
comparacion.round(1)

Empresa: The Procter & Gamble Company
Tasa efectiva de impuestos (ultimo anio): 20.8%


,FCFF (ruta EBIT),FCFF (ruta CFO),diferencia %
2026-06-30,15951.8,15841.8,0.7
2025-06-30,14949.2,14762.6,1.3
2024-06-30,15212.5,17256.8,-11.8
2023-06-30,13918.4,14385.0,-3.2


## Parte 2: proyección con supuestos declarados (4 pts)

Proyecta el FCFF a 5 años. Declara tus supuestos de crecimiento en una celda de texto y justifícalos en una o dos oraciones (historia de la empresa, sector, o crecimiento de la economía). El crecimiento perpetuo no puede superar el crecimiento nominal de largo plazo de la economía donde opera.

In [2]:
fcff_0 = float(fcff_ebit.iloc[0])          
print(f"Anio base: {fcff_ebit.index[0].year} | FCFF base = {fcff_0:,.0f} MM")
g_proyeccion = [0.02, 0.02, 0.02, 0.02, 0.02] # Tomado del promedio historico del informe 8-K  
g_perpetuo = 0.02

proy = []
f = fcff_0
for g in g_proyeccion:
    f = f * (1 + g)
    proy.append(f)

proy = pd.Series(proy, index=range(1, 6), name="FCFF proyectado")
print(proy.round(0))

Anio base: 2026 | FCFF base = 15,952 MM
1    16271.0
2    16596.0
3    16928.0
4    17267.0
5    17612.0
Name: FCFF proyectado, dtype: float64


> Se utilizo el crecimiento constante de 2% para proyectar los flujos de caja libres futuros durante 5 años, basado en el promedio histórico reportado en el informe 8-K.

## Parte 3: valoración completa (4 pts)

Con el WACC de tu tarea de la semana 2: valor presente de los flujos, valor terminal, EV, puente a equity (resta la deuda neta) y valor por acción. Reporta el peso del valor terminal en el EV y compara tu valor por acción contra el precio de mercado.

In [3]:
WACC = 0.0718272825998082886656 # Valorado en el TR

VT5 = valor_crecimiento_constante(proy.iloc[-1] * (1 + g_perpetuo), WACC, g_perpetuo)

vp_flujos = sum(cf_i / (1 + WACC) ** i for i, cf_i in proy.items())
vp_VT = VT5 / (1 + WACC) ** 5
EV = vp_flujos + vp_VT

print(f"VP de los 5 flujos      = {vp_flujos:,.0f} MM")
print(f"VP del valor terminal   = {vp_VT:,.0f} MM")
print(f"EV                      = {EV:,.0f} MM")
print(f"Peso del valor terminal = {vp_VT/EV:.0%}") # más grande que SCOO

VP de los 5 flujos      = 68,908 MM
VP del valor terminal   = 245,035 MM
EV                      = 313,944 MM
Peso del valor terminal = 78%


In [4]:
D = float(fila(bal, "Total Debt").iloc[0]) / 1e6
E = tk.fast_info["marketCap"] / 1e6

caja = float(fila(bal, "Cash And Cash Equivalents",
                  "Cash Cash Equivalents And Short Term Investments").iloc[0]) / 1e6
deuda_neta = D - caja
acciones = tk.fast_info["shares"] / 1e6

eq = EV - deuda_neta
precio_modelo = eq / acciones
precio_mercado = tk.fast_info["lastPrice"]

print(f"EV {EV:,.0f} - deuda neta {deuda_neta:,.0f} = equity {eq:,.0f} MM")
print(f"Valor por accion (modelo) = {precio_modelo:,.2f}")
print(f"Precio de mercado         = {precio_mercado:,.2f}")
print(f"Diferencia                = {precio_modelo/precio_mercado - 1:+.1%}")

EV 313,944 - deuda neta 25,084 = equity 288,860 MM
Valor por accion (modelo) = 124.37
Precio de mercado         = 151.23
Diferencia                = -17.8%


## Parte 4: veredicto profesional (2 pts)

En un párrafo: ¿tu modelo sugiere que la acción está subvaluada, sobrevaluada o bien valorada? Nombra los dos supuestos de tu modelo que más te preocupan y en qué dirección sesgarían el resultado si estuvieran mal.

> Según el modelo la accion esta sobrevaluada respecto al valor estimado. El modelo se basa principalmente en la precisión del WACC y de la tasa de crecimiento. Un crecimiento del primero implicaria un resultado menos optimista, mientras ocurre lo opuesto con el segundo.

## Parte 5: ítems tipo CFA (4 pts, 1 c/u)

**1.** Al calcular el FCFF desde la utilidad neta, el ajuste correcto por intereses es sumar:

A. el gasto por intereses completo. B. el gasto por intereses después de impuestos. C. nada, los intereses no se tocan.

**2.** Una empresa reporta FCFF = 120, interés = 30, t = 25% y endeudamiento neto = -10 (amortizó deuda). Su FCFE es:

A. 87.5. B. 97.5. C. 107.5.

**3.** El valor terminal en un DCF de dos etapas típicamente representa:

A. una fracción menor del valor total. B. entre 60 y 80 por ciento del valor total. C. exactamente el 50 por ciento del valor.

**4.** Un analista descuenta el FCFE de una empresa al WACC. Su valoración del equity estará, en general:

A. sobreestimada, porque el WACC es menor que el costo del equity. B. subestimada. C. correcta, ambas tasas son intercambiables.

**Respuestas:**

1. B. el gasto por intereses después de impuestos
2. A. $FCFE = 120 - 30*(1-0.25) -10 = 87.5$
3. B. entre 60 y 80 por ciento del valor total
4. A. sobreestimada, porque el WACC es menor que el costo del equity.

---

**Recuerda:** `Kernel  Restart & Run All` antes de entregar, y luego:

```bash
git add 03_dcf_fcff_fcfe/clase03_tarea.ipynb
git commit -m "Tarea semana 3"
git push
```